# Paid Social Impact on Brand Awareness
### Meta Ads Reach vs. Google Trends Interest — Correlation, Lag, Granger Causality & Difference-in-Differences

This notebook estimates whether paid social activity (Meta Ads) moves brand awareness
(Google Trends interest), using four complementary lenses:

1. **Correlation** — Pearson/Spearman, same-period association between reach and interest.
2. **Lag / cross-correlation** — does reach move *before* interest (consistent with ads driving
   awareness) or after (consistent with budget chasing existing demand)?
3. **Granger causality** — a formal statistical test of whether past reach improves the
   prediction of future interest, beyond interest's own history.
4. **Difference-in-Differences (DiD)** — a causal-inference comparison of countries that ran a
   campaign ("treated") against countries that didn't ("control"), before vs. after the
   campaign started. This is the closest thing to a causal effect estimate this data allows,
   since we don't have a randomized experiment.

Every analysis includes an **auto-generated plain-English interpretation** printed alongside the
numbers/chart, so you don't have to translate p-values yourself.

> ⚠️ **Correlation/Granger causality are not proof of causation.** They tell you about
> predictive relationships in the *time series*. DiD is the section that gets closest to a causal
> claim, and even that rests on the *parallel trends* assumption — checked explicitly below.


## 1. Setup

In [1]:
import sys
sys.path.insert(0, "src")

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from src import data_loader, stats_analysis, causal_inference, interpretation, viz

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 2. Load data

Point this at your real exports. Expected raw columns (exactly as described in the project spec):

**Google Trends CSV** — `brand, Country, Date, ParsedDate, interest, Geo`
**Meta Ads CSV** — `campaign name, Date, Parsed date, Reach, brand, Country`

If the files below aren't found, the notebook falls back to **synthetic demo data** (clearly
labeled) so you can see the full pipeline run end-to-end before plugging in real exports.


In [2]:
GOOGLE_TRENDS_PATH = "data/google_trends.csv"
META_ADS_PATH = "data/meta_ads.csv"

USE_SYNTHETIC = False

try:
    trends_raw = data_loader.load_google_trends(GOOGLE_TRENDS_PATH)
    ads_raw = data_loader.load_meta_ads(META_ADS_PATH)
    print(f"Loaded real data: {len(trends_raw)} trends rows, {len(ads_raw)} ads rows")
except FileNotFoundError:
    USE_SYNTHETIC = True
    print("Real data files not found under data/ — generating synthetic demo data instead.\n"
          "Drop your real CSVs at the paths above and re-run this cell to use your own data.")
    trends_raw_synth, ads_raw_synth = data_loader.generate_synthetic_data()
    # Write to disk then reload through the *real* loaders, so the exact same
    # parsing/validation code path is exercised for demo and real data alike.
    import os
    os.makedirs("data", exist_ok=True)
    trends_raw_synth.to_csv(GOOGLE_TRENDS_PATH, index=False)
    ads_raw_synth.to_csv(META_ADS_PATH, index=False)
    trends_raw = data_loader.load_google_trends(GOOGLE_TRENDS_PATH)
    ads_raw = data_loader.load_meta_ads(META_ADS_PATH)
    print(f"Synthetic data generated: {len(trends_raw)} trends rows, {len(ads_raw)} ads rows")

trends_raw.head()


Real data files not found under data/ — generating synthetic demo data instead.
Drop your real CSVs at the paths above and re-run this cell to use your own data.
Synthetic data generated: 2184 trends rows, 252 ads rows


,date,brand,country,geo,interest
0,2024-01-01,DemoBrand,Germany,DE,19.87
1,2024-01-02,DemoBrand,Germany,DE,20.16
2,2024-01-03,DemoBrand,Germany,DE,19.90
3,2024-01-04,DemoBrand,Germany,DE,24.07
4,2024-01-05,DemoBrand,Germany,DE,26.50


In [3]:
ads_raw.head()

,date,brand,geo,campaign,reach
0,2024-05-20,DemoBrand,DE,DemoBrand_DE_awareness_push,100391
1,2024-05-21,DemoBrand,DE,DemoBrand_DE_awareness_push,123992
2,2024-05-22,DemoBrand,DE,DemoBrand_DE_awareness_push,78967
3,2024-05-23,DemoBrand,DE,DemoBrand_DE_awareness_push,110252
4,2024-05-24,DemoBrand,DE,DemoBrand_DE_awareness_push,70771


## 3. Merge into a tidy analysis panel

Both datasets are aggregated to a common weekly grain (Google Trends' native resolution) and
merged on `brand + geo + date`. Weeks with no ad spend get `reach = 0` rather than being dropped,
so pre-campaign baselines are preserved.


In [4]:
panel = data_loader.merge_datasets(trends_raw, ads_raw, freq="W")
print(f"Panel shape: {panel.shape}")
print(f"Brands: {panel['brand'].unique().tolist()}")
print(f"Geos: {panel['geo'].unique().tolist()}")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")
panel.head(10)


Panel shape: (312, 6)
Brands: ['DemoBrand']
Geos: ['DE', 'ES', 'FR', 'GB', 'IT', 'US']
Date range: 2024-01-01 to 2024-12-23


,date,brand,geo,interest,reach,n_campaigns
0,2024-01-01,DemoBrand,DE,22.222857,0.0,0
1,2024-01-08,DemoBrand,DE,26.642857,0.0,0
2,2024-01-15,DemoBrand,DE,26.687143,0.0,0
3,2024-01-22,DemoBrand,DE,24.454286,0.0,0
4,2024-01-29,DemoBrand,DE,21.398571,0.0,0
5,2024-02-05,DemoBrand,DE,20.491429,0.0,0
6,2024-02-12,DemoBrand,DE,18.511429,0.0,0
7,2024-02-19,DemoBrand,DE,18.407143,0.0,0
8,2024-02-26,DemoBrand,DE,22.641429,0.0,0
9,2024-03-04,DemoBrand,DE,26.120000,0.0,0


## 4. Configure the analysis

Pick the brand and country/countries to analyze. For the DiD section you'll also need to specify
which countries were **treated** (ran a campaign) vs. **control** (didn't), and the campaign
start date.


In [5]:
BRAND = panel["brand"].unique()[0]          # <-- change to your brand
GEOS_TO_ANALYZE = sorted(panel[panel["brand"] == BRAND]["geo"].unique())

print(f"Analyzing brand: {BRAND}")
print(f"Available geos: {GEOS_TO_ANALYZE}")


Analyzing brand: DemoBrand
Available geos: ['DE', 'ES', 'FR', 'GB', 'IT', 'US']


## 5. Correlation, lag correlation & Granger causality (per country)

Run for every geo available for the selected brand. Adjust `MAX_LAG` to the number of
weeks you think a reasonable maximum delay between spend and awareness could be.


In [6]:
MAX_LAG = 6  # weeks

results = {}

for geo in GEOS_TO_ANALYZE:
    sub = panel[(panel["brand"] == BRAND) & (panel["geo"] == geo)].sort_values("date")
    if sub["reach"].sum() == 0:
        print(f"Skipping {geo}: no ad spend recorded for this brand/geo.")
        continue

    corr = stats_analysis.compute_correlation(sub["reach"], sub["interest"])
    cc = stats_analysis.cross_correlation(sub["reach"], sub["interest"], max_lag=MAX_LAG)
    bl = stats_analysis.best_lag(cc)
    granger = stats_analysis.granger_causality(sub["reach"], sub["interest"], max_lag=min(MAX_LAG, 4))

    results[geo] = {"panel": sub, "correlation": corr, "cross_corr": cc,
                     "best_lag": bl, "granger": granger}

    display(Markdown(f"### {BRAND} / {geo}"))
    fig = viz.plot_dual_axis_timeseries(panel, BRAND, geo)
    fig.show()

    display(Markdown(f"**Correlation:** {interpretation.interpret_correlation(corr, BRAND, geo)}"))

    fig = viz.plot_cross_correlation(cc, BRAND, geo, freq_label="week")
    fig.show()
    display(Markdown(f"**Lag:** {interpretation.interpret_lag(bl, BRAND, geo, freq_label='week')}"))

    if granger.get("ok"):
        fig = viz.plot_granger_pvalues(granger["p_values_by_lag"], BRAND, geo)
        fig.show()
    display(Markdown(f"**Granger causality:** {interpretation.interpret_granger(granger, BRAND, geo)}"))
    display(Markdown("---"))


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


### DemoBrand / DE

**Correlation:** [DemoBrand / DE] Reach and interest show a moderate positive correlation (Pearson r = 0.59, p = 0.000, n = 52), which is statistically significant (p < 0.05). Spearman rank correlation is 0.54 (p = 0.000), suggesting the relationship is roughly monotonic and consistent with the linear estimate.

**Lag:** [DemoBrand / DE] Strongest cross-correlation (moderate, r = 0.62) occurs at lag = -1: interest leads reach by 1 week(s) — interest moves *before* reach changes, which is more consistent with reactive budget allocation (spend chasing existing demand) than with ads driving awareness.

**Granger causality:** [DemoBrand / DE] Granger causality test: there is statistically significant evidence that past Meta Ads reach helps predict future Google Trends interest beyond interest's own history (Granger-causal at lag 2, p = 0.000) (series were differenced first to satisfy stationarity requirements).

...

### DemoBrand / US

**Correlation:** [DemoBrand / US] Reach and interest show a moderate positive correlation (Pearson r = 0.53, p = 0.000, n = 52), which is statistically significant (p < 0.05). Spearman rank correlation is 0.49 (p = 0.000), suggesting the relationship is roughly monotonic and consistent with the linear estimate.

**Lag:** [DemoBrand / US] Strongest cross-correlation (moderate, r = 0.58) occurs at lag = -1: interest leads reach by 1 week(s) — interest moves *before* reach changes, which is more consistent with reactive budget allocation (spend chasing existing demand) than with ads driving awareness.

**Granger causality:** [DemoBrand / US] Granger causality test: there is statistically significant evidence that past Meta Ads reach helps predict future Google Trends interest beyond interest's own history (Granger-causal at lag 4, p = 0.002) (series were differenced first to satisfy stationarity requirements).

---

### Summary table across countries

In [7]:
summary_rows = []
for geo, r in results.items():
    summary_rows.append({
        "geo": geo,
        "pearson_r": round(r["correlation"]["pearson_r"], 3) if r["correlation"]["n"] >= 3 else np.nan,
        "pearson_p": round(r["correlation"]["pearson_p"], 4) if r["correlation"]["n"] >= 3 else np.nan,
        "best_lag_weeks": r["best_lag"]["lag"],
        "best_lag_corr": round(r["best_lag"]["correlation"], 3) if r["best_lag"]["correlation"] == r["best_lag"]["correlation"] else np.nan,
        "granger_significant": r["granger"].get("significant_at_0.05", False),
        "granger_best_p": r["granger"].get("best_p_value", np.nan),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("pearson_r", ascending=False)
summary_df


,geo,pearson_r,pearson_p,best_lag_weeks,best_lag_corr,granger_significant,granger_best_p
1,GB,0.602,0.0000,0,0.602,True,0.0104
0,DE,0.587,0.0000,-1,0.617,True,0.0000
2,US,0.525,0.0001,-1,0.576,True,0.0019


## 6. Causal inference — Difference-in-Differences across countries

Compare **treated** countries (ran a Meta Ads campaign) to **control** countries (didn't), before
vs. after the campaign start date. The DiD coefficient is the estimated causal lift in interest
attributable to the campaign, netting out (a) baseline differences between countries and
(b) any organic trend common to all of them.

**Set these three things based on your actual campaign:**


In [8]:
TREATED_GEOS = [g for g in GEOS_TO_ANALYZE if panel[(panel.brand==BRAND) & (panel.geo==g)]["reach"].sum() > 0]
CONTROL_GEOS = [g for g in GEOS_TO_ANALYZE if g not in TREATED_GEOS]

# Campaign start = first week with non-zero reach across treated geos
TREATMENT_START = panel[(panel["brand"] == BRAND) & (panel["geo"].isin(TREATED_GEOS)) & (panel["reach"] > 0)]["date"].min()

print(f"Treated geos: {TREATED_GEOS}")
print(f"Control geos: {CONTROL_GEOS}")
print(f"Treatment start (inferred): {TREATMENT_START}")
print("\nOverride TREATED_GEOS / CONTROL_GEOS / TREATMENT_START above if this inference is wrong.")


Treated geos: ['DE', 'GB', 'US']
Control geos: ['ES', 'FR', 'IT']
Treatment start (inferred): 2024-05-20 00:00:00

Override TREATED_GEOS / CONTROL_GEOS / TREATMENT_START above if this inference is wrong.


### 6a. Parallel trends check

DiD assumes treated and control countries would have moved *in parallel* had the campaign never
happened. We can't observe that counterfactual directly, but we can check whether they moved in
parallel **before** the campaign — if they didn't, the DiD estimate below should be treated with
caution.


In [9]:
if not CONTROL_GEOS:
    print("No control geos found — DiD requires at least one country with no campaign to serve "
          "as the counterfactual. Set CONTROL_GEOS manually above.")
else:
    pre_trends = causal_inference.parallel_trends_check(panel, TREATED_GEOS, CONTROL_GEOS, TREATMENT_START)
    fig = viz.plot_parallel_trends(pre_trends)
    fig.show()


### 6b. DiD estimate

In [10]:
if CONTROL_GEOS:
    did_panel = causal_inference.build_did_panel(
        panel, TREATED_GEOS, CONTROL_GEOS, TREATMENT_START, brand=BRAND
    )
    did_result = causal_inference.run_did_regression(did_panel)
    did_sum = causal_inference.did_summary(did_result)

    fig = viz.plot_did_trends(panel[panel.brand == BRAND], TREATED_GEOS, CONTROL_GEOS, TREATMENT_START)
    fig.show()

    fig = viz.plot_did_effect_bar(did_sum)
    fig.show()

    display(Markdown(f"**Interpretation:** {interpretation.interpret_did(did_sum, TREATED_GEOS, CONTROL_GEOS)}"))

    print("\nFull regression output:")
    print(did_result.summary())


**Interpretation:** The Difference-in-Differences estimate suggests the campaign caused an average increase of 2.64 points in interest in the treated countries (DE, GB, US), relative to the counterfactual implied by the control countries (ES, FR, IT). This effect is statistically significant (p = 0.004, 95% CI [0.86, 4.42]).


Full regression output:
                            OLS Regression Results                            
Dep. Variable:               interest   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.040
Method:                 Least Squares   F-statistic:                     10.15
Date:                Wed, 01 Jul 2026   Prob (F-statistic):             0.0144
Time:                        09:16:05   Log-Likelihood:                -934.28
No. Observations:                 312   AIC:                             1877.
Df Residuals:                     308   BIC:                             1892.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         2

### 6c. Placebo test (robustness check)

Re-runs the DiD using a *fake* treatment date that falls before the real campaign, using only
pre-campaign data. If this "placebo" shows a significant effect too, treated and control groups
were likely already diverging for reasons unrelated to the campaign — a warning sign that the
main DiD estimate may not be purely causal.


In [11]:
if CONTROL_GEOS:
    pre_campaign_weeks = panel[(panel.brand == BRAND) & (panel.date < TREATMENT_START)]["date"].nunique()

    if pre_campaign_weeks >= 8:
        # Fake treatment date = halfway through the pre-campaign period
        pre_dates = sorted(panel[(panel.brand == BRAND) & (panel.date < TREATMENT_START)]["date"].unique())
        fake_start = pre_dates[len(pre_dates) // 2]

        placebo_result = causal_inference.placebo_test(
            panel, TREATED_GEOS, CONTROL_GEOS,
            fake_treatment_start=fake_start, real_treatment_start=TREATMENT_START, brand=BRAND,
        )
        display(Markdown(interpretation.interpret_placebo(placebo_result)))
    else:
        print(f"Only {pre_campaign_weeks} pre-campaign weeks available — need at least 8 for a "
              f"meaningful placebo test. Skipping.")


✅ Placebo check passed: the fake pre-campaign treatment date shows no significant effect (-0.19, p = 0.493), which is consistent with the parallel-trends assumption and supports treating the main DiD estimate as causal.

## 7. Executive summary

Auto-compiled from the results above — a quick narrative you can drop into a deck or share with
stakeholders directly.


In [12]:
lines = [f"## Executive Summary — {BRAND}\n"]

if not summary_df.empty:
    strongest = summary_df.iloc[0]
    lines.append(
        f"- Across {len(summary_df)} countries analyzed, the strongest same-period correlation "
        f"between reach and interest was in **{strongest['geo']}** (r = {strongest['pearson_r']}, "
        f"p = {strongest['pearson_p']})."
    )
    n_granger_sig = int(summary_df["granger_significant"].sum())
    lines.append(
        f"- Granger causality (reach → interest) was statistically significant in "
        f"**{n_granger_sig} of {len(summary_df)}** countries tested."
    )
    reach_leads = summary_df[summary_df["best_lag_weeks"] > 0]
    if not reach_leads.empty:
        lines.append(
            f"- In **{len(reach_leads)}** countries, the strongest lag correlation had reach "
            f"leading interest (consistent with ads driving awareness rather than the reverse)."
        )

if CONTROL_GEOS:
    lines.append(
        f"- **DiD causal estimate:** {interpretation.interpret_did(did_sum, TREATED_GEOS, CONTROL_GEOS)}"
    )
else:
    lines.append("- **DiD causal estimate:** not computed — no control countries were available.")

display(Markdown("\n".join(lines)))


## Executive Summary — DemoBrand

- Across 3 countries analyzed, the strongest same-period correlation between reach and interest was in **GB** (r = 0.602, p = 0.0).
- Granger causality (reach → interest) was statistically significant in **3 of 3** countries tested.
- **DiD causal estimate:** The Difference-in-Differences estimate suggests the campaign caused an average increase of 2.64 points in interest in the treated countries (DE, GB, US), relative to the counterfactual implied by the control countries (ES, FR, IT). This effect is statistically significant (p = 0.004, 95% CI [0.86, 4.42]).